# SLAM-ASR: An Embarrassingly Simple Approach for LLM with Strong ASR Capacity

[![arXiv](https://img.shields.io/badge/arXiv-2402.08846-b31b1b.svg)](https://arxiv.org/abs/2402.08846)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanersoy/slam-asr-reimplement/blob/main/slam_asr_colab.ipynb)

Bu notebook, **arXiv:2402.08846** çalışmasının Google Colab (ücretsiz T4 veya A100 GPU) üzerinde uçtan uca çalıştırılması için hazırlanmıştır.

### 🌟 Temel Mimari Prensipleri
1. **Speech Foundation Encoder (FROZEN):** HuBERT, WavLM veya Whisper ağırlıkları dondurulur.
2. **Frame Downsampler ($k=5$):** 50 Hz olan konuşma öznitelikleri, ardışık 5 çerçevenin birleştirilmesiyle **10 Hz'e** indirgenir.
3. **Trainable Projector (TRAINABLE):** Yalnızca tek katmanlı lineer katman (veya 2 katmanlı MLP) eğitilir (~18M - 21M parametre).
4. **Causal LLM Backbone (FROZEN):** Vicuna-7B veya TinyLlama dondurulmuş olarak kullanılır.
5. **Format:** `USER: <S> <Prompt> ASSISTANT: <Transcript><eos>` (Kayıp yalnızca hedef metin tokenları üzerinden hesaplanır).

## 1. GPU ve Donanım Kontrolü

In [1]:
# GPU kullanılabilirliğini kontrol edin (Colab'da Runtime -> Change runtime type -> T4 GPU seçili olmalıdır)
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Sun Oct 11 21:02:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Proje Dosyalarını Çekme ve Bağımlılıkları Yükleme

Proje dosyalarını Colab'a aktarmak için aşağıdaki **iki yöntemden birini** kullanabilirsiniz:

- **Yöntem 1 (Önerilen):** GitHub deposundan doğrudan klonlamak (`git clone`).
- **Yöntem 2:** Bilgisayarınızdaki `slam-asr-reimplement.zip` dosyasını Colab'ın sol dosya paneline sürükleyip açmak (`unzip`).

In [2]:
import os

# Eğer çalışma dizininde slam_asr klasörü yoksa GitHub'dan klonla
if not os.path.exists("slam_asr") and not os.path.exists("slam-asr-reimplement/slam_asr"):
    print("[*] Proje GitHub'dan klonlanıyor...")
    !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd slam-asr-reimplement
elif os.path.exists("slam-asr-reimplement"):
    %cd slam-asr-reimplement
else:
    print("[*] Proje dosyaları zaten aktif dizinde mevcut.")

# Gerekli kütüphaneleri requirements.txt üzerinden yükleyin
!pip install -q -r requirements.txt

[*] Proje GitHub'dan klonlanıyor...
Cloning into 'slam-asr-reimplement'...
remote: Enumerating objects: 711, done.
remote: Counting objects: 100% (711/711), done.
remote: Compressing objects: 100% (626/626), done.
remote: Total 711 (delta 78), reused 701 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (711/711), 1.98 MiB | 6.95 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/slam-asr-reimplement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.6 MB/s eta 0:00:00


## 3. Kod Dosyalarının Doğrulanması ve Birim Testler

In [3]:
# Testleri koşturarak modüllerin (Projector, Collator, Metrics, Model) doğru çalıştığını teyit edin
!python run_tests.py

test_variable_length_waveform_collation (test_collator.TestCollator.test_variable_length_waveform_collation) ... ok
test_whisper_mel_collation (test_collator.TestCollator.test_whisper_mel_collation) ... ok
test_compute_cer (test_metrics.TestMetrics.test_compute_cer) ... ok
test_compute_wer (test_metrics.TestMetrics.test_compute_wer) ... ok
test_levenshtein_distance (test_metrics.TestMetrics.test_levenshtein_distance) ... ok
test_text_normalization (test_metrics.TestMetrics.test_text_normalization) ... ok
test_downsampler_discard_remainder (test_projector.TestProjector.test_downsampler_discard_remainder) ... ok
test_downsampler_pad_remainder (test_projector.TestProjector.test_downsampler_pad_remainder) ... ok
test_linear_projector (test_projector.TestProjector.test_linear_projector) ... ok
test_mlp_projector (test_projector.TestProjector.test_mlp_projector) ... ok
test_speech_projector_end_to_end (test_projector.TestProjector.test_speech_projector_end_to_end) ... ok
test_forward_pass_an

## 4. Hızlı Doğrulama: Sentetik Veri ile Test Eğitimi
Sistemin forward-backward döngüsünün, gradyan izolasyonunun ve erken durdurma mekanizmasının çalıştığını saniyeler içinde görmek için sentetik ses verisi üretip test edebilirsiniz.

In [4]:
# 1. Sentetik toy verisini üretin
!python scripts/create_toy_dataset.py

# 2. Hızlı debug eğitimi başlatın (CPU/T4 üzerinde anında çalışır)
!python train.py --config config/train_toy_debug.yaml --max_steps 20


[*] Generated toy dataset:
    Train: /content/slam-asr-reimplement/data/sample_train.jsonl (4 samples)
    Val:   /content/slam-asr-reimplement/data/sample_val.jsonl (1 samples)
    Audio: /content/slam-asr-reimplement/data/toy_audio
[*] Initializing SLAM-ASR with config: config/train_toy_debug.yaml
config.json: 100% 1.39k/1.39k [00:00<00:00, 4.21MB/s]

pytorch_model.bin: downloading bytes:  12% 44.9M/378M [00:01<00:09, 33.8MB/s, 3.00MB/s  ]
pytorch_model.bin: reconstructing file:  14% 51.8M/378M [00:01<00:12, 26.8MB/s]
pytorch_model.bin: downloading bytes:  27% 102M/378M [00:02<00:03, 84.2MB/s, 7.89MB/s  ] 
pytorch_model.bin: downloading bytes:  31% 118M/378M [00:02<00:03, 82.6MB/s, 10.3MB/s  ]
pytorch_model.bin: reconstructing file:  43% 161M/378M [00:02<00:02, 88.1MB/s, 12.6MB/s  ]
pytorch_model.bin: downloading bytes:  37% 141M/378M [00:02<00:02, 91.9MB/s, 11.8MB/s  ]
pytorch_model.bin: downloading bytes:  46% 174M/378M [00:02<00:01, 119MB/s, 13.9MB/s  ]
pytorch_model.bin: downloa

## 5. Gerçek LibriSpeech Verisi Hazırlığı (Hugging Face Üzerinden)
Hugging Face `openslr/librispeech_asr` veri setinden küçük bir dilim indirip manifestoları otomatik hazırlayalım.

In [5]:
import os
import json
import soundfile as sf
from datasets import load_dataset
from tqdm import tqdm

os.makedirs("data/colab_audio", exist_ok=True)

def prepare_hf_split(split_name, output_path, max_samples=300):
    print(f"[*] Downloading '{split_name}' from openslr/librispeech_asr (max {max_samples} samples)...")
    # Yeni huggingface_hub sürümlerinde namespace zorunludur: 'openslr/librispeech_asr'
    try:
        ds = load_dataset("openslr/librispeech_asr", "clean", split=split_name, streaming=True)
    except Exception:
        ds = load_dataset("distil-whisper/librispeech_asr", "clean", split=split_name, streaming=True)

    records = []
    for i, item in enumerate(tqdm(ds, total=max_samples)):
        if i >= max_samples:
            break
        key = item.get("id") or f"sample_{i}"
        audio_info = item["audio"]
        wav_path = os.path.abspath(f"data/colab_audio/{key}.wav")
        if not os.path.exists(wav_path):
            sf.write(wav_path, audio_info["array"], audio_info["sampling_rate"])

        records.append({
            "key": str(key),
            "source": wav_path,
            "target": item["text"],
            "prompt": "Transcribe speech to text."
        })

    with open(output_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"[*] Saved {len(records)} samples to {output_path}")

# Eğitim için 300, doğrulama için 50 örnek hazırlayalım
prepare_hf_split("train.100", "data/colab_train.jsonl", max_samples=300)
prepare_hf_split("validation", "data/colab_val.jsonl", max_samples=50)

[*] Downloading 'train.100' from openslr/librispeech_asr (max 300 samples)...


README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

100%|██████████| 300/300 [00:10<00:00, 28.94it/s]


[*] Saved 300 samples to data/colab_train.jsonl
[*] Downloading 'validation' from openslr/librispeech_asr (max 50 samples)...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:02<00:00, 17.23it/s]

[*] Saved 50 samples to data/colab_val.jsonl


## 6. Colab T4 Üzerinde SLAM-ASR Eğitimi

İki seçenek sunulmuştur:
1. **TinyLlama-1.1B + HuBERT-Base (`config/colab_t4_tinyllama.yaml`):** Ultra hafif (~3.8 GB VRAM), Colab T4 üzerinde son derece hızlı eğitilir.
2. **Vicuna-7B (4-Bit NF4) + HuBERT-Large (`config/colab_t4_vicuna_4bit.yaml`):** Makaledeki 7B chat modelini 4-bit kuantizasyonla T4 VRAM'ine sığdırır (~6.5 GB VRAM).

In [6]:
# Konfigürasyonu güncelleyerek Colab verisiyle eğitimi başlatın
# Seçenek 1: TinyLlama ile hızlı eğitim (Önerilen test başlangıcı)
!python train.py \
    --config config/colab_t4_tinyllama.yaml \
    --train_manifest data/colab_train.jsonl \
    --val_manifest data/colab_val.jsonl \
    --output_dir ./outputs/slam_colab_run \
    --max_steps 300 \
    --batch_size 4


[*] Initializing SLAM-ASR with config: config/colab_t4_tinyllama.yaml
Loading weights: 100% 211/211 [00:00<00:00, 15358.95it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:04<00:00, 42.71it/s]
/content/slam-asr-reimplement/slam_asr/train/trainer.py:82: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.args.fp16 and self.device.type == "cuda")
[*] Starting SLAM-ASR Training on device: cuda
[*] Trainable parameters: 7,866,368 / 1,202,286,464 (0.65%)
Training:   0% 0/300 [00:00<?, ?it/s]/content/slam-asr-reimplement/slam_asr/train/trainer.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=self.amp_dtype):
/content/slam-asr-reimplement/slam_asr/train/trainer.py:154: UserWarning: Detected cal

## 7. Model Değerlendirmesi (WER ve CER Hesaplama)

In [7]:
# En iyi checkpoint üzerinde test manifestosunu değerlendirin
!python evaluate.py \
    --config config/colab_t4_tinyllama.yaml \
    --checkpoint outputs/slam_colab_run/best_checkpoint \
    --manifest data/colab_val.jsonl \
    --beam_size 4 \
    --batch_size 4

[*] Loading model on cuda...
Loading weights: 100% 211/211 [00:00<00:00, 18018.16it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:02<00:00, 97.09it/s]
[*] Loading projector weights from: outputs/slam_colab_run/best_checkpoint/projector.pt
[*] Evaluating 50 samples with beam_size=4...
Evaluating:   0% 0/13 [00:00<?, ?it/s]/content/slam-asr-reimplement/evaluate.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=amp_dtype):
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluating:   8% 1/13 [00:12<02:25, 12.13s/it][transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set.

## 8. İnteraktif Çıkarım (Ses Dinleme ve Transcribe)

In [8]:
import IPython.display as ipd
import soundfile as sf

# Örnek bir ses dosyasını dinleyin
audio_path = "data/toy_audio/toy_1.wav"
ipd.display(ipd.Audio(audio_path))

# Modeli kullanarak sesi yazıya dökün
!python inference.py \
    --audio data/toy_audio/toy_1.wav \
    --config config/colab_t4_tinyllama.yaml \
    --checkpoint outputs/slam_colab_run/best_checkpoint/projector.pt \
    --prompt "Transcribe speech to text." \
    --beam_size 4

[*] Loading SLAM-ASR on cuda...
Loading weights: 100% 211/211 [00:00<00:00, 15158.75it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:02<00:00, 99.69it/s] 
[*] Loading projector weights from: outputs/slam_colab_run/best_checkpoint/projector.pt
[*] Processing audio: data/toy_audio/toy_1.wav
[*] Transcribing (beam_size=4)...
/content/slam-asr-reimplement/inference.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=amp_dtype):
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

Audio:          data/toy_audio/toy_1.wav
Prompt:         Transcribe speech to text.
Transcription:  the quick brown fox jumps over the